# PFC -> FNO: physics-penalty ablation (diagnostic)

**Goal:** find out *why* the grain run stopped early at epoch 17 and *which* point-seed
failures are real. We train the SAME grain-rich dataset twice -- once with the free-energy
dissipation penalty (`physics_weight=0.1`) and once without (`0.0`) -- with `patience=30`
so the full training curve emerges. Both models are evaluated with the new
**phase-insensitive spectral metric**, which separates benign lattice phase-offsets
(structure correct, pixels shifted) from genuinely wrong structure.

Set the runtime first: **Runtime -> Change runtime type -> T4 GPU**, then `Runtime -> Run all`.
A Google Drive popup appears at the first cell -- approve it.

### 1. Mount Drive + clone repo (with the new spectral metric) + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy pyyaml matplotlib tensorboard

### 2. Confirm a GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU OK:", torch.cuda.get_device_name(0))

## Part A - Get the grain-rich dataset
Reuses `pfc_dataset_grains.zip` from your Drive if it's there (instant); otherwise
regenerates the 90-run dataset (`n_seeds=30`) on CPU (~1 min).

In [ ]:
import os
drive_zip = '/content/drive/MyDrive/pfc_dataset_grains.zip'
if os.path.exists(drive_zip):
    print('Found dataset on Drive -- unzipping (no regeneration needed).')
    !cp "$drive_zip" .
    !unzip -o -q pfc_dataset_grains.zip
else:
    print('No dataset on Drive -- generating it now.')
    !python generate_dataset.py --sweep --config pfc_config_grains.yaml --parallel --max-workers 2
    !zip -r -q pfc_dataset_grains.zip data_pfc_grains
    !cp pfc_dataset_grains.zip /content/drive/MyDrive/

In [ ]:
import pandas as pd
m = pd.read_csv('data_pfc_grains/manifest.csv')
print(m['status'].value_counts())
print(f"{len(m)} runs | seed types: {sorted(m['seed_type'].unique())}")

## Part B - Configure the ablation
Point the config at the grain dataset, keep `split_by: config` (no ensemble leakage),
and confirm `patience` is now 30. Everything else (delta-weighted loss, 4-step rollout,
modes=24, enforce_mass) is already set in `config.yaml`.

In [ ]:
import yaml
with open('config.yaml') as f:
    base = yaml.safe_load(f)
base['data']['data_dir'] = 'data_pfc_grains'
base['data']['split_by'] = 'config'
with open('config.yaml', 'w') as f:
    yaml.dump(base, f, default_flow_style=False, sort_keys=False)
print('data       :', base['data'])
print('patience   :', base['train']['patience'], '(was 15)')
print('loss/roll  :', {k: base['train'][k] for k in ['loss_weight_mode','loss_alpha','rollout_steps']})
print('physics_wt :', base['train']['physics_weight'], '(overridden per run below)')

### Helper: train + evaluate one configuration
Writes a per-run `logging.out_dir` and `eval.out_dir` so the two runs never collide.
The data split is identical across runs (same `split_seed`, same dataset), so the test
set is the same and the comparison is fair.

In [ ]:
import yaml, subprocess, shutil, os

def run_experiment(physics_weight, tag):
    with open('config.yaml') as f:
        cfg = yaml.safe_load(f)
    cfg['data']['data_dir'] = 'data_pfc_grains'
    cfg['data']['split_by'] = 'config'
    cfg['train']['physics_weight'] = physics_weight
    out_dir = f'runs/ablation_{tag}'
    cfg['logging']['out_dir'] = out_dir
    cfg['eval']['out_dir'] = f'{out_dir}/eval'
    with open('config.yaml', 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    # PYTHONUNBUFFERED + python -u so train/eval progress streams LIVE into the
    # cell output (plain subprocess block-buffers and looks frozen for minutes).
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    print(f'\n===== TRAIN [{tag}]  physics_weight={physics_weight}  patience={cfg["train"]["patience"]} =====', flush=True)
    subprocess.run(['python', '-u', 'train_fno.py', '--config', 'config.yaml'], check=True, env=env)
    ckpt = f'{out_dir}/best.pt'
    shutil.copy(ckpt, f'/content/drive/MyDrive/fno_grains_{tag}.pt')
    print(f'\n===== EVAL [{tag}] =====', flush=True)
    subprocess.run(['python', '-u', 'evaluate_fno.py', '--config', 'config.yaml', '--checkpoint', ckpt], check=True, env=env)
    return out_dir

### Run A - physics penalty ON (`physics_weight = 0.1`)
This reproduces the run that produced `Fno Grains Best.pt`, but with `patience=30`.

In [ ]:
run_experiment(0.1, 'physics')

### Run B - physics penalty OFF (`physics_weight = 0.0`)
The ablation: identical in every way except the free-energy term is removed.

In [ ]:
run_experiment(0.0, 'nophysics')

## Part C - Compare
**How to read this:**
- If B trains well past epoch 17 while A still stalls early -> the physics penalty was
  the blocker (re-add it at ~0.01).
- If both plateau at the same epoch -> the penalty is innocent; the limit is the
  dataset / phase-offset basin.
- `mean_spectral_mse` << `mean_rollout_mse` and a high `phase_offset_trajs` count means
  most of the error is benign lattice registration, not broken structure.

In [ ]:
import pandas as pd, torch
rows = []
for tag in ['physics','nophysics']:
    df = pd.read_csv(f'runs/ablation_{tag}/eval/per_trajectory.csv')
    ck = torch.load(f'runs/ablation_{tag}/best.pt', map_location='cpu', weights_only=False)
    rows.append({
        'run': tag,
        'physics_weight': 0.1 if tag=='physics' else 0.0,
        'best_epoch': ck.get('epoch'),
        'best_val_loss': ck.get('val_loss'),
        'mean_rollout_mse': round(float(df['rollout_mse'].mean()), 6),
        'mean_spectral_mse': round(float(df['spectral_mse'].mean()), 6),
        'phase_offset_trajs': int(df['phase_offset'].sum()),
        'n_traj': len(df),
    })
summary = pd.DataFrame(rows)
display(summary)

In [ ]:
# Per-seed-type rollout MSE, side by side
import pandas as pd
frames = []
for tag in ['physics','nophysics']:
    df = pd.read_csv(f'runs/ablation_{tag}/eval/per_trajectory.csv')
    g = df.groupby('seed_type')[['rollout_mse','spectral_mse']].mean().add_suffix(f'_{tag}')
    frames.append(g)
display(pd.concat(frames, axis=1))

### Rollout figures (worst trajectories) for each run

In [ ]:
from IPython.display import Image, display
import glob
for tag in ['physics','nophysics']:
    print(f'================= {tag} =================')
    for png in sorted(glob.glob(f'runs/ablation_{tag}/eval/*.png')):
        display(Image(png))

### Save both evaluations to Drive

In [ ]:
for tag in ['physics','nophysics']:
    !zip -r -q fno_grains_eval_{tag}.zip runs/ablation_{tag}/eval
    !cp fno_grains_eval_{tag}.zip /content/drive/MyDrive/
print('Saved to Drive: fno_grains_physics.pt, fno_grains_nophysics.pt,')
print('               fno_grains_eval_physics.zip, fno_grains_eval_nophysics.zip')